# Pre-processing News Data

## Bài toán
Dữ liệu gồm n văn bản phân vào 10 chủ đề khác nhau. Cần biễu diễn mỗi văn bản dưới dạng một vector số thể hiện cho nội dụng của văn bản đó.

## Sử dụng phương pháp mã hóa: TF-IDF
Cho tập gồm $n$ văn bản: $D = \{d_1, d_2, ... d_n\}$. Tập từ điển tương ứng được xây dựng từ $n$ văn bản này có độ dài là $m$
- Xét văn bản $d$ có $|d|$ từ và $t$ là một từ trong $d$. Mã hóa tf-idf của $t$ trong văn bản $d$ được biểu diễn:
\begin{equation}
    \begin{split}
        \text{tf}_{t, d} &= \frac{f_t}{|d|} \\
        \text{idf}_{t, d} &= \log\frac{n}{n_t}, \ \ \ \ n_t = |\{d\in D: t\in d\}| \\
        \text{tf-idf}_{t d} &= \text{tf}_{t, d} \times \text{idf}_{t, d}
    \end{split}
\end{equation}

- Khi đó văn bản $d$ được mã hóa là một vector $m$ chiều. Các từ xuất hiện trong d sẽ được thay bằng giá trị tf-idf tương ứng. Các từ không xuất hiện trong $d$ thì thay là 0

# Các bước làm

## Chuẩn bị các thư viện cần thiết

In [ ]:
# pyvi đã được cài sẵn trong Docker image (xem requirements.txt) nên không cần cài lại.
# Trên Google Colab thì mới cần chạy:
# !pip install pyvi

In [1]:
import os
import matplotlib.pyplot as plt
import numpy as np

from sklearn.datasets import load_files
from pyvi import ViTokenizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.pipeline import Pipeline

%matplotlib inline

In [ ]:
# Chạy cục bộ bằng Docker/Jupyter nên không cần kết nối Google Drive.
# (Trên Google Colab thì mới cần hai dòng dưới đây)
# from google.colab import drive
# drive.mount('/content/drive')

## Load dữ liệu từ thư mục

Cấu trúc thư mục như sau

- data/news_vnexpress/

    - Kinh tế:
        - bài báo 1.txt
        - bài báo 2.txt
    - Pháp luật
        - bài báo 3.txt
        - bài báo 4.txt

In [2]:
#Có thể chỉnh lại đường dẫn thư mục cho phù hợp
INPUT = 'news_vnexpress'  # thư mục chứa 10 chủ đề, nằm cùng cấp với notebook
os.makedirs("images", exist_ok=True)  # thư mục lưu các hình ảnh trong quá trình huấn luyện và đánh giá

In [3]:
# statistics
print('Các nhãn và số văn bản tương ứng trong dữ liệu')
print('----------------------------------------------')
n = 0
for label in os.listdir(INPUT):
    print(f'{label}: {len(os.listdir(os.path.join(INPUT, label)))}')
    n += len(os.listdir(os.path.join(INPUT, label)))

print('-------------------------')
print(f"Tổng số văn bản: {n}")

Các nhãn và số văn bản tương ứng trong dữ liệu
----------------------------------------------
doi-song: 120
du-lich: 54
giai-tri: 201
giao-duc: 105
khoa-hoc: 144
kinh-doanh: 262
phap-luat: 59
suc-khoe: 162
the-thao: 173
thoi-su: 59
-------------------------
Tổng số văn bản: 1339


In [4]:
# load data
data_train = load_files(container_path=INPUT, encoding="utf-8")
print('mapping:')
for i in range(len(data_train.target_names)):
    print(f'{data_train.target_names[i]} - {i}')

print('--------------------------')
print(data_train.filenames[0:1])
# print(data_train.data[0:1])
print(data_train.target[0:1])
print(data_train.data[0:1])

print("\nTổng số  văn bản: {}" .format( len(data_train.filenames)))

mapping:
doi-song - 0
du-lich - 1
giai-tri - 2
giao-duc - 3
khoa-hoc - 4
kinh-doanh - 5
phap-luat - 6
suc-khoe - 7
the-thao - 8
thoi-su - 9
--------------------------
['news_vnexpress/khoa-hoc/00133.txt']
[4]
['Mời độc giả đặt câu hỏi tại đây\n']

Tổng số  văn bản: 1339


## Chuyển dữ liệu dạng text về ma trận (n x m) bằng TF-IDF

* Bạn cần viết đoạn mã tương ứng trong cell bên dưới. Theo các bước được gợi ý

In [5]:
# load dữ liệu các stopwords
with open("vietnamese-stopwords.txt", encoding="utf-8") as f:
    # mỗi dòng là một từ dừng; thay khoảng trắng bằng "_" để khớp với token của ViTokenizer
    stopwords = [line.strip().replace(" ", "_") for line in f if line.strip()]

# Chuyển hoá dữ liệu text về dạng vector TF
#     - loại bỏ từ dừng
#     - sinh từ điển
module_count_vector = CountVectorizer(stop_words=stopwords)
module_tfidf_transformer = TfidfTransformer()

# Hàm thực hiện chuyển đổi dữ liệu text thành dữ liệu số dạng ma trận
# Input: Dữ liệu 2 chiều dạng numpy.array, mảng nhãn id dạng numpy.array
model_preprocess = Pipeline([
    ('vect', module_count_vector),        # đếm số lần xuất hiện của từ -> ma trận TF
    ('tfidf', module_tfidf_transformer),  # chuyển TF -> TF-IDF
])

# Tách từ tiếng Việt cho từng văn bản trước khi vector hoá
data_tokenized = [ViTokenizer.tokenize(text) for text in data_train.data]
data_preprocessed = model_preprocess.fit_transform(data_tokenized, data_train.target)

X = data_preprocessed # thuoc tinh
Y = data_train.target #nhan

print(f"\nSố lượng từ trong từ điển: {len(module_count_vector.vocabulary_)}")
print(f"Kích thước dữ liệu sau khi xử lý: {X.shape}")
print(f"Kích thước nhãn tương ứng: {Y.shape}")


Số lượng từ trong từ điển: 24351
Kích thước dữ liệu sau khi xử lý: (1339, 24351)
Kích thước nhãn tương ứng: (1339,)


In [6]:
print(X[100].toarray())
print(Y[100])

[[0. 0. 0. ... 0. 0. 0.]]
5


In [7]:
print(X[100]) #Sau khi xử lí, dữ liệu được lưu dưới dạng ma trận thưa như sau:

  (0, 24168)	0.05212619241149833
  (0, 24137)	0.03283876747053921
  (0, 23988)	0.0792831693702859
  (0, 23961)	0.03047417521593614
  (0, 23847)	0.03682936736098612
  (0, 23733)	0.028199616254834484
  (0, 23690)	0.043379946548368964
  (0, 23634)	0.03722580562017943
  (0, 23627)	0.10205686822329726
  (0, 23619)	0.03722580562017943
  (0, 23503)	0.17779640781809294
  (0, 23459)	0.0314666913746395
  (0, 23457)	0.040906127133630094
  (0, 23397)	0.03455093498858012
  (0, 23385)	0.052456483316520044
  (0, 23298)	0.07529256947983899
  (0, 23258)	0.027817824705684238
  (0, 23238)	0.026931200395042997
  (0, 22997)	0.05331834821051844
  (0, 22694)	0.02163284432551321
  (0, 22586)	0.08889820390904647
  (0, 22540)	0.16419383735269605
  (0, 22533)	0.03241828835079914
  (0, 22504)	0.029339277314491482
  (0, 22489)	0.025901791695506493
  :	:
  (0, 3641)	0.02631231276425595
  (0, 3330)	0.03455093498858012
  (0, 3186)	0.04571364440301038
  (0, 3178)	0.03305896949935565
  (0, 3040)	0.029743417719199833
  